# Milestone 5 — Cross-dataset synthesis & cross-validation

Combines the per-dataset aggregated tables from notebooks 01–04 (Allen scRNA, Allen
MERFISH, Vizgen, Zhuang) into a single **evidence table** keyed by (cell_type,
brain_area, gene), with detection rates, cross-dataset concordance, and a
**confidence tier** per row. Level-agnostic: works at whatever `cell_type_level`
the config uses.

Independence: Allen MERFISH *imputed* genes are predicted from Allen scRNA, so they
are tracked as *supporting* evidence only — not counted toward independent
concordance. Run notebooks 01–04 first so the aggregate parquets exist (re-run them
after the recent code changes so they carry `frac_expressing`/`n_cells`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import load_config, resolve_output_dir, restrict_config_to_genes, start_run
from src.data_loaders import get_abc_cache, merfish_gene_source_map
from src import synthesis as syn
from src.utils import print_path

In [ ]:
# Unified panel (receptors + excitability). Synthesis carries the 'category' column.
CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
EXPLORATION_ROOT = resolve_output_dir(cfg=config)
config["_dataset_modality"] = "synthesis"

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="synthesis",
    exploration_root=EXPLORATION_ROOT,
    notebook="05_synthesis",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Gene panel key: {config['_gene_panel_key']}")
print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
print(f"Run dir: {OUTPUT_DIR}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

aggregates, sources = syn.gather_dataset_aggregates(
    config, exploration_root=EXPLORATION_ROOT,
)
print(f"\nDatasets loaded: {sorted(aggregates)}")
for key, df in aggregates.items():
    has_frac = "frac_expressing" in df.columns
    print(f"  {key}: {len(df):,} rows, frac_expressing={'yes' if has_frac else 'MISSING (re-run nb)'}")
    print(f"    from {sources.get(key)}")

In [ ]:
# Allen MERFISH measured-vs-imputed provenance (drives independence accounting).
allen_sources = {}
if "allen_merfish" in aggregates:
    genes_present = sorted(aggregates["allen_merfish"]["gene"].unique())
    allen_sources = merfish_gene_source_map(cache, genes_present, config)
    n_imp = sum(1 for v in allen_sources.values() if v == "imputed")
    print(f"Allen MERFISH genes: {len(allen_sources)} ({n_imp} imputed, supporting-only)")

evidence = syn.build_evidence_table(
    aggregates, config, allen_gene_sources=allen_sources,
)
print(f"\nEvidence rows (cell_type × region × gene): {len(evidence):,}")
print("\nConfidence tier counts:")
print(evidence["confidence_tier"].value_counts())

In [ ]:
if config["output"].get("save_processed_data", True):
    pq = OUTPUT_DIR / "evidence_table.parquet"
    csv = OUTPUT_DIR / "evidence_table.csv"
    evidence.to_parquet(pq, index=False)
    evidence.to_csv(csv, index=False)
    print(f"Saved {pq}")
    print(f"Saved {csv}")

## Per-target reports (all cell types × brain areas)

The **evidence table** above already contains every `(cell_type, brain_area, gene)` row.
The cells below export **one organized report folder per target** under
`targets/{cell_type}__{region}/`:

- `evidence_target.csv` — full target table (all tiers)
- `statement.txt` — cross-validated claim scaffold
- `receptors/*.csv`, `excitability/*.csv` — per-family tables (high/medium only)
- `figures/evidence_by_family.pdf` — one page per family (PDF only)

**Regions** come from `config['brain_areas']` (CCF acronyms). Allen MERFISH, Vizgen,
and Zhuang aggregates are already mapped to those acronyms in notebooks 02–04; Allen
scRNA expression is **broadcast** to each region (same values, region-specific rows).

**Cell types** are narrowed by `cell_type_name_filter` in the config (substring match,
e.g. `L2/3` and `L5`). Set it to `[]` to include every supertype present in the
evidence table.

A manifest `targets/_index.csv` lists every target, tier counts, and paths. Set
`SKIP_UNEXPRESSED=True` to skip targets with no high/medium genes (faster; fewer
placeholder-only PDFs).

In [ ]:
# Export scope — defaults: all config brain_areas × cell types matching cell_type_name_filter
TARGET_REGIONS = None       # None → config['brain_areas']; or e.g. ["VISpm", "VISp"]
TARGET_CELL_TYPES = None  # None → filter from config; or explicit list
SKIP_UNEXPRESSED = True   # skip targets with zero high/medium genes

targets = syn.list_report_targets(
    evidence,
    config,
    regions=TARGET_REGIONS,
    cell_types=TARGET_CELL_TYPES,
)
print(f"Targets to export: {len(targets)}")
if targets:
    by_region = pd.Series([r for _, r in targets]).value_counts().sort_index()
    print("\nPer region:")
    print(by_region.to_string())
    print(f"\nFirst 10: {targets[:10]}")

In [ ]:
index_df = syn.export_all_target_reports(
    evidence,
    config,
    output_dir=OUTPUT_DIR,
    regions=TARGET_REGIONS,
    cell_types=TARGET_CELL_TYPES,
    skip_unexpressed=SKIP_UNEXPRESSED,
)
if not index_df.empty:
    print_path("Manifest:", OUTPUT_DIR / "targets" / "_index.csv")
    display(index_df.sort_values(["brain_area", "cell_type"]).head(20))

In [ ]:
# Optional: inspect one exported target (first non-skipped row in the manifest)
if index_df.empty:
    print("No targets exported.")
else:
    row = index_df[~index_df["skipped"]].iloc[0]
    PREVIEW_CELL_TYPE = row["cell_type"]
    PREVIEW_REGION = row["brain_area"]
    print(f"Preview: {PREVIEW_CELL_TYPE} × {PREVIEW_REGION}")

    summary = syn.summarize_target(
        evidence, cell_type=PREVIEW_CELL_TYPE, region=PREVIEW_REGION,
    )
    print(syn.statement_scaffold(summary))

    tev = syn.target_evidence(
        evidence, cell_type=PREVIEW_CELL_TYPE, region=PREVIEW_REGION,
    )
    target_cols = [
        "gene", "family", "category", "confidence_tier",
        "n_independent_measured_detections", "supporting_imputed_detection",
    ]
    display(tev[target_cols].head(40))
    print_path("Target dir:", row["target_dir"])
    print_path("PDF:", row["pdf"])

## Functional landscape — resonance & neuromodulator modules

**Targets:** `(supertype, brain_area)` — sourced from run folders named
`*_{cell_type_level}_{dataset}*` (see level report below; independent of
`config['cell_type_level']` used by the synthesis cells above).

**Expression:** high/medium confidence tier only. Spatial datasets preferred
(MERFISH → Vizgen → Zhuang → scRNA) for VISp vs V2M area contrasts.

**Modules:** coupling-centric neuromodulator axes (Gi/Gq/Gs) + subthreshold
resonance gene sets from `excitability_genes.md` / `receptor_excitability.md`.

**Ephys fixture:** `data/test_experimental_resonance.csv` — **one row per recorded
cell** (not aggregated). Default n=20 cells per coarse_type × area group; L5 ET /
VISp is bimodal (`subcluster_hint`: ET_low / ET_high) to demo within-type functional
clusters for a future milestone.

In [ ]:
from src import functional_analysis as fa

func_level = fa.functional_config(config)["cell_type_level"]
func_evidence, func_sources, level_report = fa.prepare_functional_evidence(
    config,
    exploration_root=EXPLORATION_ROOT,
    allen_gene_sources=allen_sources,
)
fa.print_level_source_report(level_report)
display(level_report)
print(f"\nFunctional evidence rows ({func_level}): {len(func_evidence):,}")

modules = fa.load_functional_modules(config)
module_genes = sorted({g for m in modules.values() for g in m["genes"]})
print(f"Functional modules: {len(modules)}")
for name, spec in modules.items():
    print(f"  {name}: {len(spec['genes'])} genes — {spec.get('description', '')}")

expr_matrix, expr_prov = fa.build_target_expression_matrix(
    func_evidence, config, genes=module_genes,
)
print(f"\nTargets ({func_level} × region): {len(expr_matrix):,}")
print(f"Genes with ≥1 value: {(expr_matrix.notna().any(axis=0)).sum()} / {len(module_genes)}")
print("\nDataset usage (high/medium tier picks):")
if not expr_prov.empty:
    print(expr_prov.groupby("dataset").size().sort_values(ascending=False).to_string())

In [ ]:
fa_cfg = fa.functional_config(config)
module_scores = fa.compute_module_scores(
    expr_matrix,
    modules,
    min_genes=fa_cfg["min_module_genes"],
)
clusters = fa.cluster_targets(module_scores, config)
embedding = fa.joint_embedding(module_scores, config)

display(module_scores.head(15))
if not clusters.empty:
    print(f"\nClusters: {clusters.value_counts().sort_index().to_dict()}")

In [ ]:
func_dir = OUTPUT_DIR / "functional"
func_dir.mkdir(parents=True, exist_ok=True)

# VISp vs V2M module-score contrast (V2M − VISp, per supertype)
vis_contrast = fa.compute_vis_group_contrast(module_scores, config)
delta_cols = [c for c in vis_contrast.columns if c.startswith("delta_")]
print(f"VISp–V2M contrast: {len(vis_contrast)} cell types with both groups")
if not vis_contrast.empty:
    display(vis_contrast[["cell_type", "coarse_type"] + delta_cols].head(20))

fa.plot_module_score_heatmap(
    module_scores, config,
    output_path=func_dir / "module_scores_heatmap.pdf",
    title=f"Functional module scores ({func_level} × region)",
)
fa.plot_cluster_dendrogram(
    module_scores, config,
    output_path=func_dir / "module_cluster_dendrogram.pdf",
)
fa.plot_joint_embedding(
    embedding, module_scores, config,
    color_module="subthreshold_resonance_core",
    output_path=func_dir / "joint_embedding_resonance_color.pdf",
    title="Joint embedding — colour = subthreshold_resonance_core",
)
fa.plot_joint_embedding(
    embedding, module_scores, config,
    color_module=None,
    output_path=func_dir / "joint_embedding_area_group.pdf",
    title="Joint embedding — colour = VISp vs V2M",
)
print_path("Functional figures:", func_dir)

In [ ]:
# Experimental resonance — single-cell CSV (replace path in config when ready)
ephys_path = fa.resolve_experimental_resonance_path(config, PROJECT_ROOT)
print_path("Experimental resonance CSV:", ephys_path)
experimental = fa.load_experimental_resonance(ephys_path)
print(f"Ephys cells loaded: {len(experimental):,} (one row per cell)")
display(fa.experimental_within_type_summary(experimental))

link_table = fa.link_targets_to_experimental(module_scores, experimental, config)
paths = fa.export_functional_landscape(
    expr_matrix, module_scores, embedding, clusters,
    expr_prov, link_table, OUTPUT_DIR,
    vis_contrast=vis_contrast,
    level_report=level_report,
)
paths.update(fa.export_experimental_sidecar(experimental, OUTPUT_DIR))
for key, p in paths.items():
    print_path(key, p)

display(link_table.filter(regex="cell_type|coarse|ephys_", axis=1).head(10))